# Pipeline Medallion — Bronze / Prata / Ouro

Pipeline completo de processamento de dados de devices seguindo a arquitetura **Medallion**:

| Camada | Bucket | Descricao |
|--------|--------|-----------|
| **Landing** | `s3a://landing/` | Dados brutos em JSON (origem) |
| **Bronze** | `s3a://bronze/` | Dados crus + metadados de ingestao (Delta) |
| **Prata** | `s3a://prata/` | Dados limpos, tipados e deduplicados (Delta) |
| **Ouro** | `s3a://ouro/` | Agregacoes de negocio prontas para consumo (Delta) |

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import (
    col, current_timestamp, lit, trim, upper, lower,
    from_unixtime, count, desc
)
from pyspark.sql.types import TimestampType
from delta.tables import DeltaTable

spark = SparkSession \
    .builder \
    .appName("medallion-pipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} conectado ao cluster")

Spark 3.5.5 conectado ao cluster


---
## Camada Bronze

Ingestao dos dados brutos do **landing** com adicao de metadados de controle:
- `_ingestion_timestamp`: quando o dado foi ingerido
- `_source_format`: formato de origem do arquivo

Os dados sao gravados em **Delta Lake** no modo `append`, simulando chegada continua de dados.

In [15]:
# =============================================
# BRONZE: Ingestao crua + metadados
# =============================================

LANDING_PATH = "s3a://landing/*.json"
BRONZE_PATH = "s3a://bronze/device/raw"

df_landing = spark.read \
    .format("json") \
    .option("inferSchema", "true") \
    .json(LANDING_PATH)

print(f"Registros lidos do landing: {df_landing.count()}")

df_bronze = df_landing \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_format", lit("json"))

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .save(BRONZE_PATH)

print(f"Bronze gravado em {BRONZE_PATH}")

Registros lidos do landing: 200
Bronze gravado em s3a://bronze/device/raw


In [16]:
dt = DeltaTable.forPath(spark, 's3a://bronze/device/raw')

In [29]:
spark.sql("""
DESCRIBE HISTORY delta.`s3a://bronze/device/raw` LIMIT 5
""").toPandas()

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,3,2026-02-27 01:48:57,None,None,WRITE,"{'mode': 'Overwrite', 'partitionBy': '[]'}",None,None,None,2.0,Serializable,False,"{'numOutputRows': '200', 'numOutputBytes': '22...",None,Apache-Spark/3.5.5 Delta-Lake/3.2.0
1,2,2026-02-27 01:43:05,None,None,WRITE,"{'mode': 'Overwrite', 'partitionBy': '[]'}",None,None,None,1.0,Serializable,False,"{'numOutputRows': '200', 'numOutputBytes': '22...",None,Apache-Spark/3.5.5 Delta-Lake/3.2.0
2,1,2026-02-27 01:42:22,None,None,WRITE,"{'mode': 'Overwrite', 'partitionBy': '[]'}",None,None,None,0.0,Serializable,False,"{'numOutputRows': '200', 'numOutputBytes': '22...",None,Apache-Spark/3.5.5 Delta-Lake/3.2.0
3,0,2026-02-25 03:03:47,None,None,WRITE,"{'mode': 'Overwrite', 'partitionBy': '[]'}",None,None,None,NaN,Serializable,False,"{'numOutputRows': '200', 'numOutputBytes': '22...",None,Apache-Spark/3.5.5 Delta-Lake/3.2.0


In [18]:
#df_v10 = spark.read.format("delta").option("versionAsOf", 10).table("s3a://bronze/device/raw")

In [20]:
def listar_versoes_delta(table_identifier: str = None, table_path: str = None, limit: int = 1000):
    """
    Lista versões disponíveis (histórico) de uma tabela Delta.
    Use OU table_identifier (ex: 'catalog.schema.tabela' ou 'schema.tabela')
    OU table_path (ex: 's3://bucket/caminho' ou '/mnt/delta/tabela').
    """
    if (table_identifier is None) == (table_path is None):
        raise ValueError("Informe exatamente um: table_identifier OU table_path.")

    if table_identifier:
        history_df = spark.sql(f"DESCRIBE HISTORY {table_identifier}")
    else:
        history_df = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`")

    # Colunas típicas: version, timestamp, operation, operationParameters, userName, userId, job, notebook, clusterId, readVersion, isolationLevel, isBlindAppend, ...
    # Nem todas aparecem em todos os ambientes.
    cols = history_df.columns

    # Seleciona um subconjunto seguro e comum
    select_cols = [c for c in ["version", "timestamp", "operation", "userName", "userId", "clusterId", "readVersion", "isolationLevel"] if c in cols]
    out = (history_df
           .select(*select_cols)
           .orderBy(F.col("version").asc())
           .limit(limit))

    # Resumo das versões disponíveis
    versions = (history_df
                .select(F.min("version").alias("min_version"),
                        F.max("version").alias("max_version"),
                        F.countDistinct("version").alias("qtd_versions"))
               )

    return out, versions


In [44]:
# EXEMPLO 1: por nome da tabela (Metastore/Unity Catalog)
#hist, resumo = listar_versoes_delta(table_identifier="meu_schema.minha_tabela")
#hist.show(truncate=False)
#resumo.show(truncate=False)

# EXEMPLO 2: por caminho (tabela Delta em storage)
hist, resumo = listar_versoes_delta(table_path="s3a://bronze/device/raw")
hist.show(truncate=False)
resumo.show(truncate=False)

+-------+-------------------+---------+--------+------+---------+-----------+--------------+
|version|timestamp          |operation|userName|userId|clusterId|readVersion|isolationLevel|
+-------+-------------------+---------+--------+------+---------+-----------+--------------+
|0      |2026-02-25 03:03:47|WRITE    |NULL    |NULL  |NULL     |NULL       |Serializable  |
|1      |2026-02-27 01:42:22|WRITE    |NULL    |NULL  |NULL     |0          |Serializable  |
|2      |2026-02-27 01:43:05|WRITE    |NULL    |NULL  |NULL     |1          |Serializable  |
|3      |2026-02-27 01:48:57|WRITE    |NULL    |NULL  |NULL     |2          |Serializable  |
|4      |2026-02-27 02:00:18|RESTORE  |NULL    |NULL  |NULL     |3          |Serializable  |
|5      |2026-02-27 02:00:44|RESTORE  |NULL    |NULL  |NULL     |4          |Serializable  |
+-------+-------------------+---------+--------+------+---------+-----------+--------------+

+-----------+-----------+------------+
|min_version|max_version|qtd_v

In [40]:
dt = DeltaTable.forPath(spark, 's3a://bronze/device/raw')

dt.toDF().toPandas()

,build_number,dt_current_timestamp,id,manufacturer,model,platform,serial_number,uid,user_id,version,_ingestion_timestamp,_source_format
0,365,1654630765114,7938,Xiamomi,Samsung Galaxy S9,Windows 8,Yr9Vt13BlgvXO9zgJTPuCLv6F82r5S,aeb52d1e-5a00-412f-a20b-483085a8e6a9,1992,255,2026-02-27 01:48:56.956937,json
1,340,1654630765114,8981,Huawei,Xiaomi Pocophone F1,Windows 8.1,Kl2ZroV9a,cd4b4510-9cd4-4f2d-b68e-683aabb4f9f6,1464,393,2026-02-27 01:48:56.956937,json
2,230,1654630765114,635,Lenovo,OnePlus 6,Android,ToFVWLzGTJhQxAaJlDDn,accee742-a5c9-4724-916e-c4196d0d6374,6853,713,2026-02-27 01:48:56.956937,json
3,423,1654630765114,1320,OnePlus,iPhone 4S,webOS,Yr9Vt13BlgvXO9zgJTPuCLv6F82r5S,bac54e13-33c3-412c-9a1f-47ea4bd9befe,3945,597,2026-02-27 01:48:56.956937,json
4,18,1654630765114,3550,Huawei,Samsung Galaxy S6 Edge,Android OS,T6UuMUTani3VGY4vXGia,a81af7fc-bb8c-4407-96dc-6558a055ed23,6740,63,2026-02-27 01:48:56.956937,json
...,...,...,...,...,...,...,...,...,...,...,...,...
195,95,1654630947030,8222,OnePlus,iPhone 3GS,Windows 10 Mobile,pEekWH7zGxVITv6NTa5KHjLSwr5Ie4,e432e7f4-36a8-4fe0-8fa5-38ed22cfe2f3,148,738,2026-02-27 01:48:56.956937,json
196,481,1654630947030,8805,Huawei,Google Pixel,Danger OS,39gPmcOKpwhDezLdiIOZ7SH89Pbjp4,915f2522-f043-4dd3-9316-6d23a121a4b9,4429,332,2026-02-27 01:48:56.956937,json
197,480,1654630947030,2131,Xiamomi,Google Pixel 2,Windows 8,VMTnd2mMQWvjbtNcZh7UIdULKb1mMo,49076083-493b-456d-adce-9d86a1536913,549,915,2026-02-27 01:48:56.956937,json
198,278,1654630947030,9118,ASUS,OnePlus 3,Windows 10 Mobile,UVr864F8zUbyYOAUd4cFOW9hpsZuGn,5d01728f-ddb8-4355-be7f-4b19e4ec4781,9501,600,2026-02-27 01:48:56.956937,json


In [43]:
# sql
spark.sql("""
RESTORE TABLE delta.`s3a://bronze/device/raw`
TO VERSION AS OF 1""").show()

26/02/27 02:00:41 WARN DAGScheduler: Broadcasting large task binary with size 1078.1 KiB


+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|table_size_after_restore|num_of_files_after_restore|num_removed_files|num_restored_files|removed_files_size|restored_files_size|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|                   22087|                         2|                0|                 0|                 0|                  0|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+



In [27]:
(spark.read
     .format("delta")
     .option("versionAsOf", 1)
     .load('s3a://bronze/device/raw')
).toPandas()

,build_number,dt_current_timestamp,id,manufacturer,model,platform,serial_number,uid,user_id,version,_ingestion_timestamp,_source_format
0,365,1654630765114,7938,Xiamomi,Samsung Galaxy S9,Windows 8,Yr9Vt13BlgvXO9zgJTPuCLv6F82r5S,aeb52d1e-5a00-412f-a20b-483085a8e6a9,1992,255,2026-02-27 01:42:19.188177,json
1,340,1654630765114,8981,Huawei,Xiaomi Pocophone F1,Windows 8.1,Kl2ZroV9a,cd4b4510-9cd4-4f2d-b68e-683aabb4f9f6,1464,393,2026-02-27 01:42:19.188177,json
2,230,1654630765114,635,Lenovo,OnePlus 6,Android,ToFVWLzGTJhQxAaJlDDn,accee742-a5c9-4724-916e-c4196d0d6374,6853,713,2026-02-27 01:42:19.188177,json
3,423,1654630765114,1320,OnePlus,iPhone 4S,webOS,Yr9Vt13BlgvXO9zgJTPuCLv6F82r5S,bac54e13-33c3-412c-9a1f-47ea4bd9befe,3945,597,2026-02-27 01:42:19.188177,json
4,18,1654630765114,3550,Huawei,Samsung Galaxy S6 Edge,Android OS,T6UuMUTani3VGY4vXGia,a81af7fc-bb8c-4407-96dc-6558a055ed23,6740,63,2026-02-27 01:42:19.188177,json
...,...,...,...,...,...,...,...,...,...,...,...,...
195,95,1654630947030,8222,OnePlus,iPhone 3GS,Windows 10 Mobile,pEekWH7zGxVITv6NTa5KHjLSwr5Ie4,e432e7f4-36a8-4fe0-8fa5-38ed22cfe2f3,148,738,2026-02-27 01:42:19.188177,json
196,481,1654630947030,8805,Huawei,Google Pixel,Danger OS,39gPmcOKpwhDezLdiIOZ7SH89Pbjp4,915f2522-f043-4dd3-9316-6d23a121a4b9,4429,332,2026-02-27 01:42:19.188177,json
197,480,1654630947030,2131,Xiamomi,Google Pixel 2,Windows 8,VMTnd2mMQWvjbtNcZh7UIdULKb1mMo,49076083-493b-456d-adce-9d86a1536913,549,915,2026-02-27 01:42:19.188177,json
198,278,1654630947030,9118,ASUS,OnePlus 3,Windows 10 Mobile,UVr864F8zUbyYOAUd4cFOW9hpsZuGn,5d01728f-ddb8-4355-be7f-4b19e4ec4781,9501,600,2026-02-27 01:42:19.188177,json


In [3]:
# Validacao Bronze
df_bronze_check = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze — total de registros: {df_bronze_check.count()}")
df_bronze_check.printSchema()
df_bronze_check.show(5, truncate=False)

Bronze — total de registros: 200
root
 |-- build_number: long (nullable = true)
 |-- dt_current_timestamp: long (nullable = true)
 |-- id: long (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- uid: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- version: long (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_format: string (nullable = true)



+------------+--------------------+----+------------+----------------------+-----------+------------------------------+------------------------------------+-------+-------+--------------------------+--------------+
|build_number|dt_current_timestamp|id  |manufacturer|model                 |platform   |serial_number                 |uid                                 |user_id|version|_ingestion_timestamp      |_source_format|
+------------+--------------------+----+------------+----------------------+-----------+------------------------------+------------------------------------+-------+-------+--------------------------+--------------+
|365         |1654630765114       |7938|Xiamomi     |Samsung Galaxy S9     |Windows 8  |Yr9Vt13BlgvXO9zgJTPuCLv6F82r5S|aeb52d1e-5a00-412f-a20b-483085a8e6a9|1992   |255    |2026-02-27 01:43:03.762666|json          |
|340         |1654630765114       |8981|Huawei      |Xiaomi Pocophone F1   |Windows 8.1|Kl2ZroV9a                     |cd4b4510-9cd4-4f2d-b6

---
## Camada Prata (Silver)

Limpeza e padronizacao dos dados:
- Converter `dt_current_timestamp` (epoch ms) para tipo `timestamp`
- Remover duplicatas por `id`
- Filtrar registros com `id` ou `uid` nulos
- Padronizar `manufacturer` (UPPER) e `platform` (lower)
- Renomear colunas para nomes mais claros

In [ ]:
# =============================================
# PRATA: Limpeza e padronizacao
# =============================================

PRATA_PATH = "s3a://prata/device"

df_bronze_raw = spark.read.format("delta").load(BRONZE_PATH)
bronze_count = df_bronze_raw.count()

df_prata = df_bronze_raw \
    .filter(col("id").isNotNull() & col("uid").isNotNull()) \
    .dropDuplicates(["id"]) \
    .withColumn("event_timestamp", (col("dt_current_timestamp") / 1000).cast(TimestampType())) \
    .withColumn("manufacturer", upper(trim(col("manufacturer")))) \
    .withColumn("platform", lower(trim(col("platform")))) \
    .select(
        col("id").alias("device_id"),
        col("uid"),
        col("user_id"),
        col("manufacturer"),
        col("model"),
        col("platform"),
        col("version"),
        col("build_number"),
        col("serial_number"),
        col("event_timestamp"),
        col("_ingestion_timestamp")
    )

df_prata.write \
    .format("delta") \
    .mode("overwrite") \
    .save(PRATA_PATH)

prata_count = df_prata.count()
print(f"Bronze: {bronze_count} registros")
print(f"Prata:  {prata_count} registros")
print(f"Removidos: {bronze_count - prata_count} (duplicatas + nulos)")

In [ ]:
# Validacao Prata
df_prata_check = spark.read.format("delta").load(PRATA_PATH)
print(f"Prata — total de registros: {df_prata_check.count()}")
df_prata_check.printSchema()
df_prata_check.show(5, truncate=False)

In [ ]:
dt = DeltaTable.forPath(spark, 's3a://prata/device')

In [ ]:
dt.toDF().toPandas()

---
## Camada Ouro (Gold)

Agregacoes de negocio prontas para consumo pelo Dremio / Metabase:
1. **Devices por fabricante** — ranking dos maiores fabricantes
2. **Devices por plataforma** — distribuicao por SO
3. **Resumo fabricante x plataforma** — tabela cruzada

In [ ]:
# =============================================
# OURO: Agregacoes de negocio
# =============================================

OURO_FABRICANTE_PATH = "s3a://ouro/device/por_fabricante"
OURO_PLATAFORMA_PATH = "s3a://ouro/device/por_plataforma"
OURO_RESUMO_PATH = "s3a://ouro/device/fabricante_plataforma"

df_silver = spark.read.format("delta").load(PRATA_PATH)

# --- 1. Devices por fabricante ---
df_por_fabricante = df_silver \
    .groupBy("manufacturer") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_por_fabricante.write \
    .format("delta") \
    .mode("overwrite") \
    .save(OURO_FABRICANTE_PATH)

print("Devices por fabricante:")
df_por_fabricante.show(truncate=False)

In [ ]:
# --- 2. Devices por plataforma ---
df_por_plataforma = df_silver \
    .groupBy("platform") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_por_plataforma.write \
    .format("delta") \
    .mode("overwrite") \
    .save(OURO_PLATAFORMA_PATH)

print("Devices por plataforma:")
df_por_plataforma.show(truncate=False)

In [ ]:
# --- 3. Resumo fabricante x plataforma ---
df_resumo = df_silver \
    .groupBy("manufacturer", "platform") \
    .agg(count("*").alias("total_devices")) \
    .orderBy(desc("total_devices"))

df_resumo.write \
    .format("delta") \
    .mode("overwrite") \
    .save(OURO_RESUMO_PATH)

print("Resumo fabricante x plataforma (top 20):")
df_resumo.show(20, truncate=False)

---
## Resumo do Pipeline

In [ ]:
# Contagem final de cada camada
landing_count = spark.read.json("s3a://landing/*.json").count()
bronze_count = spark.read.format("delta").load(BRONZE_PATH).count()
prata_count = spark.read.format("delta").load(PRATA_PATH).count()
ouro_fab = spark.read.format("delta").load(OURO_FABRICANTE_PATH).count()
ouro_plat = spark.read.format("delta").load(OURO_PLATAFORMA_PATH).count()
ouro_resumo = spark.read.format("delta").load(OURO_RESUMO_PATH).count()

print("=" * 50)
print("RESUMO DO PIPELINE MEDALLION")
print("=" * 50)
print(f"Landing  (JSON):           {landing_count} registros")
print(f"Bronze   (Delta raw):      {bronze_count} registros")
print(f"Prata    (Delta limpo):    {prata_count} registros")
print(f"Ouro     - por fabricante: {ouro_fab} linhas")
print(f"Ouro     - por plataforma: {ouro_plat} linhas")
print(f"Ouro     - resumo cruzado: {ouro_resumo} linhas")
print("=" * 50)

In [ ]:
spark.stop()
print("SparkSession encerrada.")